# 🌀 Apache Airflow — Workflow Automation
## Python Ecosystem Tutorial Series — Module 16 of 18

**Author:** Himanshu Goel | [hgoelgithub.github.io](https://hgoelgithub.github.io)

---

| | |
|---|---|
| **Library** | 🌀 Apache Airflow |
| **Domain** | Workflow Automation |
| **Dataset** | ML pipeline DAG |
| **Module** | 16 of 18 |

**What you will learn:**

1. What Airflow is and why it exists
2. Core concepts and data structures
3. Hands-on code with real data
4. Visualisations and interpretation
5. When to use it and alternatives

```bash
# Install required libraries
pip install apache-airflow
```

## Quick Reference Card

| Code | What it does |
|------|--------------|
| `DAG(dag_id, schedule)` | Define pipeline |
| `PythonOperator()` | Run Python function |
| `BashOperator()` | Run shell command |
| `>>` | Set task dependency |
| `xcom_push/pull` | Pass data between tasks |

# 16. 🌀 Apache Airflow — Workflow Automation
> **Python + Apache Airflow = Workflow Automation**

Airflow schedules and monitors complex data pipelines as **DAGs**
(Directed Acyclic Graphs). Run daily ETL jobs, ML retraining, data quality checks.

**Key concepts:** DAG, tasks, operators, scheduling, XCom, sensors, backfill

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import numpy as np
from datetime import datetime, timedelta

print("Apache Airflow DAG Code")
print("(pip install apache-airflow — runs as a web server)")
print()

airflow_code = """
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from airflow.sensors.filesystem import FileSensor
from datetime import datetime, timedelta
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# ── DAG definition ───────────────────────────────────────────────────────────
default_args = {
    "owner":            "himanshu",
    "retries":          2,
    "retry_delay":      timedelta(minutes=5),
    "email_on_failure": True,
    "email":            ["alert@lab.org"],
}

with DAG(
    dag_id       = "toxicology_ml_pipeline",
    schedule     = "@daily",              # run every day
    start_date   = datetime(2025, 1, 1),
    catchup      = False,                  # don't backfill old dates
    default_args = default_args,
    tags         = ["ml","toxicology"],
) as dag:

    # ── Task 1: Check if new data arrived ────────────────────────────────────
    wait_for_data = FileSensor(
        task_id    = "wait_for_data",
        filepath   = "/data/new_chemicals/*.csv",
        timeout    = 3600,  # wait up to 1 hour
        poke_interval = 60,
    )

    # ── Task 2: Extract & validate data ──────────────────────────────────────
    def extract_and_validate(**context):
        df = pd.read_csv("/data/new_chemicals/latest.csv")
        assert df["smiles"].notna().all(), "Missing SMILES!"
        assert df["mw"].between(50, 1000).all(), "MW out of range!"
        df.to_parquet("/data/processed/chemicals.parquet")
        context["ti"].xcom_push(key="n_chemicals", value=len(df))  # pass to next task
        print(f"Validated {len(df)} chemicals")

    extract = PythonOperator(
        task_id         = "extract_and_validate",
        python_callable = extract_and_validate,
    )

    # ── Task 3: Compute features ──────────────────────────────────────────────
    compute_features = BashOperator(
        task_id      = "compute_features",
        bash_command = "python /scripts/featurize.py --input /data/processed/chemicals.parquet",
    )

    # ── Task 4: Retrain ML model ──────────────────────────────────────────────
    def retrain_model(**context):
        n = context["ti"].xcom_pull(task_ids="extract_and_validate", key="n_chemicals")
        print(f"Retraining on {n} new chemicals")
        df = pd.read_parquet("/data/features/X.parquet")
        # ... train model, evaluate, save ...
        rf = RandomForestClassifier(n_estimators=200, random_state=42)
        # rf.fit(X, y)
        # joblib.dump(rf, "/models/toxicity_v2.pkl")

    retrain = PythonOperator(
        task_id="retrain_model", python_callable=retrain_model
    )

    # ── Task 5: Push to API ────────────────────────────────────────────────────
    deploy = BashOperator(
        task_id      = "deploy_model",
        bash_command = "curl -X POST http://api.lab.org/models/update -d @/models/meta.json",
    )

    # ── Define task order (the DAG) ───────────────────────────────────────────
    wait_for_data >> extract >> compute_features >> retrain >> deploy
    #       ↓              ↓              ↓              ↓         ↓
    #  Sensor         Python         BashOperator   Python    BashOperator
"""
print(airflow_code)

# ── Visualise the DAG ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Panel 1: DAG graph
ax = axes[0]
ax.set_xlim(0, 14); ax.set_ylim(0, 5); ax.axis("off")
ax.set_facecolor("#1F2937")

tasks = [
    (1.2, 2.5, "wait_for_data\n(FileSensor)",    "#F1C40F"),
    (3.8, 2.5, "extract_\nvalidate\n(Python)",   "#3498DB"),
    (6.4, 2.5, "compute_\nfeatures\n(Bash)",     "#27AE60"),
    (9.0, 2.5, "retrain_\nmodel\n(Python)",      "#8E44AD"),
    (11.6,2.5, "deploy_\nmodel\n(Bash)",         "#E74C3C"),
]
for x, y, label, col in tasks:
    rect = mpatches.FancyBboxPatch((x-1.0,y-0.85),2.0,1.7,
                                    boxstyle="round,pad=0.1",
                                    facecolor=col,edgecolor="white",alpha=0.9,lw=2)
    ax.add_patch(rect)
    ax.text(x, y, label, ha="center", va="center", fontsize=7.5,
             fontweight="bold", color="white")

for i in range(len(tasks)-1):
    ax.annotate("", xy=(tasks[i+1][0]-1.0,2.5), xytext=(tasks[i][0]+1.0,2.5),
                 arrowprops=dict(arrowstyle="->",color="white",lw=2.5))

ax.text(7, 4.4, "Airflow DAG: toxicology_ml_pipeline  •  schedule: @daily",
         ha="center", fontsize=10, fontweight="bold", color="white")
ax.text(7, 0.4, "Tasks run left → right | failure → retry 2× | email alert on failure",
         ha="center", fontsize=8, color="#9CA3AF")

# Panel 2: Schedule timeline (Gantt-style)
ax2 = axes[1]
days = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
task_names = ["wait_for_data","extract","compute_features","retrain","deploy"]
duration_mins = [15, 8, 25, 45, 3]
task_colors   = ["#F1C40F","#3498DB","#27AE60","#8E44AD","#E74C3C"]

for day_i, day in enumerate(days):
    t_start = 0.0
    for t_i, (tname, dur, col) in enumerate(zip(task_names, duration_mins, task_colors)):
        ax2.broken_barh([(day_i*60+t_start, dur)], (t_i-0.4, 0.8),
                         facecolors=col, alpha=0.85, edgecolor="white")
        t_start += dur

ax2.set_yticks(range(len(task_names)))
ax2.set_yticklabels(task_names, fontsize=8)
ax2.set_xticks([i*60+48 for i in range(7)])
ax2.set_xticklabels(days)
ax2.set_xlabel("Time →")
ax2.set_title("Weekly Pipeline Schedule (Gantt view)\nDaily run ~96 min total", fontweight="bold")
ax2.grid(True, alpha=0.2, axis="x")

plt.suptitle("Apache Airflow — Workflow Automation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("airflow_dag.png", dpi=120, bbox_inches="tight")
plt.show()

## Deep Dive: Apache Airflow

### DAG Fundamentals
A DAG (Directed Acyclic Graph) is a dependency specification for tasks. "Directed" means arrows show which task feeds into which. "Acyclic" means no circular dependencies (guarantees the pipeline terminates). It's just a flowchart written in Python.

### Task Definition Patterns
```python
# Python function as a task
def my_function(**context):
    # context["ti"] = TaskInstance (for XCom)
    # context["ds"] = execution date string ("2024-01-15")
    # context["params"] = DAG-level parameters
    pass

task = PythonOperator(task_id="my_task", python_callable=my_function)
```

### Dependency Operators
```python
a >> b          # b runs after a
a >> [b, c]     # b and c run after a (in parallel!)
[a, b] >> c     # c runs after both a and b complete
a >> b >> c >> d  # linear chain
```

### Key Airflow Concepts
- **Schedule**: when the DAG runs (cron expression or @daily, @hourly)
- **DAG Run**: one execution of the DAG (e.g., the January 15 run)
- **Task Instance**: one execution of a task within a DAG Run
- **XCom**: key-value store for passing small data between tasks
- **Backfill**: retroactively run the DAG for historical dates

### Production Best Practices
Store large data in S3 or a database — not in XCom (XCom is for small values like row counts or file paths). Use sensors (FileSensor, S3KeySensor) to wait for upstream data rather than hardcoding delays.


## ✅ Key Takeaways — 🌀 Apache Airflow

1. Airflow is cron + visibility + retry + dependencies — the upgrade every data pipeline needs
2. Lazy operators: >> means 'depends on', [a, b] >> c means 'wait for both'
3. XCom passes small metadata between tasks; large data goes to S3 or databases
4. The Airflow UI is essential — always check the Graph View and Task Logs

---
*Next: Continue to Module 17 of 18 in the Python Ecosystem Tutorial Series*  
*Portfolio: [hgoelgithub.github.io](https://hgoelgithub.github.io)*